## `port`, `targetPort`, `nodePort` — the three numbers

Service manifests carry up to three port numbers. New users blur them; the CKA expects precision.

```yaml
spec:
  ports:
    - port: 80          # what the Service listens on (cluster-internal)
      targetPort: 5678  # what the pod's container listens on
      nodePort: 31234   # NodePort type only — port opened on every node
      name: http        # required when there's more than one port
```

The flow:

```
in-cluster client  :port→  Service  :targetPort→  Pod (container listens here)
external client    :nodePort→  node  → (same kube-proxy rules)   # NodePort only
```

- **`port`** — the port the Service exposes to in-cluster clients. **Required.**
- **`targetPort`** — the pod port traffic is forwarded to. Defaults to `port` if omitted. Can be a number *or* a **named port** from the pod spec (`targetPort: http` matches the container's `ports[*].name: http`). Named ports are safer — the container can move its port and you edit one file.
- **`nodePort`** — only with `type: NodePort` (or `LoadBalancer`, which implies one). Opens this port on *every* node, range `30000–32767`; auto-assigned if omitted.

The classic mistake: setting `targetPort` to the host port, or to `port`, when the container actually listens elsewhere. **Always read `targetPort` from the container's `containerPort`.** On our map these three numbers are all properties of the one **Service** chip — the translation layer between what clients dial and where the container actually listens.